# Modern Data Pipelines with PGAA

- Postgres Distributed (PGD) provides high-availability and scalability by design but it also comes with the Postgres Analytics Accelerator (PGAA)
- PGAA replicates data from PGD seamlessly to Iceberg without code or an external CDC tool
- WarehousePG also uses PGAA to query Iceberg directly for Analytics and Machine Learning 

This demo shows EDB PGD handling live transactional claims traffic, automatically tiering that data to Iceberg on S3, and WarehousePG reading it straight from object storage to transform and score it with in-database MADlib ML. This modernizes your data pipeline from the legacy ETL pattern to a modern data pipeline.


## Setup Requirements
1. Either use the `vpc.yaml` in CloudFormation to deploy a VPC and subnets or use an existing VPC and subnets in AWS.
2. Deploy EDB PGD in AWS using `pgd.yaml` using CloudFormation. This template has been preconfigured for this demo.
3. After successfully deploying PGD with CloudFormation, use the Output tab to get the public ip address of either node and then `ssh` to that node with your private key. e.g. `ssh -i my-private-key.pem rocky@192.1.1.1`
4. Copy `01_claims.sql`, `02_replicate_claims.sql`, `setup.sh`, and `claims.py` to one of the PGD nodes.
5. Switch to the postgres user `sudo su - postgres` and then run `./setup.sh`.
6. Start loading data with `python3 claims.py`


## Init
Connect to WarehousePG and set environment variables for the demo.

In [1]:
# Environment variables
storage_location = "pgaa-demo"
bucket_name = "demo-pgaa"
region = "us-east-2"

from sqlalchemy import create_engine
PGUSER="gpadmin"
PGHOST="cdw"
PGPORT="5432"
PGDATABASE="dev"
conn = create_engine(f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}")

# If running elsewhere without trust auth, set a password and use this instead:
# PGPASSWORD="your_password_here"
# conn = create_engine(f"postgresql://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}")

%reload_ext sql
%sql conn

## PGAA Configuration
Configure PGAA in WarehousePG so it can read Iceberg tables stored in S3.

In [2]:
%%sql
ALTER USER current_user SET pgaa.executor_engine = 'seafowl';
ALTER USER current_user SET pgaa.enable_direct_scan = off;
SELECT pgfs.delete_storage_location('{{storage_location}}');
SELECT pgfs.create_storage_location('{{storage_location}}', 's3://{{bucket_name}}', '{"region": "{{region}}"}');


Running query in 'postgresql://gpadmin@cdw:5432/dev'

1 rows affected.

1 rows affected.

create_storage_location
pgaa-demo


## EDW
Create the EDW tables for claims. This has the historic data that is used in conjuction with new data that is added to Iceberg via PGAA and PostgreSQL.

In [3]:
%%sql
DROP SCHEMA IF EXISTS claims_demo CASCADE;
CREATE SCHEMA IF NOT EXISTS claims_demo;

-- ---------------------------------------------------------------------
-- groups: the insurance group (employer/plan sponsor) a member belongs to
-- ---------------------------------------------------------------------
CREATE TABLE claims_demo.groups (
    group_id            INTEGER         NOT NULL,
    group_name          VARCHAR(100)    NOT NULL,
    group_type          VARCHAR(30),        -- Employer, Individual, Government, Association
    funding_type        VARCHAR(20),        -- Fully Insured, Self Funded
    industry_sic        VARCHAR(10),
    state               CHAR(2),
    effective_date      DATE            NOT NULL,
    termination_date    DATE
)
DISTRIBUTED BY (group_id);

-- ---------------------------------------------------------------------
-- members: the insured person (subscriber or dependent)
-- ---------------------------------------------------------------------
CREATE TABLE claims_demo.members (
    member_id           INTEGER         NOT NULL,
    group_id            INTEGER         NOT NULL,
    subscriber_id       VARCHAR(20)     NOT NULL,   -- ties dependents to their subscriber
    first_name          VARCHAR(50),
    last_name           VARCHAR(50),
    date_of_birth       DATE,
    gender              CHAR(1),
    relationship_code   VARCHAR(15),        -- Subscriber, Spouse, Dependent
    plan_type           VARCHAR(10),        -- HMO, PPO, EPO, HDHP
    address_line1       VARCHAR(100),
    city                VARCHAR(50),
    state               CHAR(2),
    zip_code            VARCHAR(10),
    effective_date      DATE            NOT NULL,
    termination_date    DATE
)
DISTRIBUTED BY (member_id);

-- ---------------------------------------------------------------------
-- providers: the healthcare provider (individual or facility)
-- ---------------------------------------------------------------------
CREATE TABLE claims_demo.providers (
    provider_id         INTEGER         NOT NULL,
    npi                 VARCHAR(10)     NOT NULL,   -- National Provider Identifier
    provider_name       VARCHAR(100)    NOT NULL,
    provider_type       VARCHAR(20),        -- Individual, Facility, Group
    taxonomy_code       VARCHAR(10),
    specialty_desc      VARCHAR(60),
    tax_id              VARCHAR(15),
    address_line1       VARCHAR(100),
    city                VARCHAR(50),
    state               CHAR(2),
    zip_code            VARCHAR(10),
    network_status      VARCHAR(15),        -- In-Network, Out-of-Network
    effective_date      DATE            NOT NULL
)
DISTRIBUTED BY (provider_id);

-- ---------------------------------------------------------------------
-- diagnosis: ICD-10-CM diagnosis code reference table
-- ---------------------------------------------------------------------
CREATE TABLE claims_demo.diagnosis (
    diagnosis_id        INTEGER         NOT NULL,
    diagnosis_code      VARCHAR(10)     NOT NULL,   -- ICD-10-CM
    diagnosis_desc      VARCHAR(255)    NOT NULL,
    diagnosis_category  VARCHAR(60),
    chronic_flag        BOOLEAN         DEFAULT FALSE
)
DISTRIBUTED REPLICATED;

-- ---------------------------------------------------------------------
-- procedures: CPT/HCPCS procedure code reference table
-- ---------------------------------------------------------------------
CREATE TABLE claims_demo.procedures (
    procedure_id        INTEGER         NOT NULL,
    procedure_code      VARCHAR(10)     NOT NULL,   -- CPT / HCPCS
    procedure_desc      VARCHAR(255)    NOT NULL,
    procedure_category  VARCHAR(60),
    code_type           VARCHAR(10),        -- CPT, HCPCS
    is_preventative     BOOLEAN         NOT NULL DEFAULT FALSE
)
DISTRIBUTED REPLICATED;

-- ---------------------------------------------------------------------
-- claim_header: one row per medical claim
-- ---------------------------------------------------------------------
CREATE TABLE claims_demo.claim_header (
    claim_id                    BIGINT          NOT NULL,
    member_id                   INTEGER         NOT NULL,
    group_id                    INTEGER         NOT NULL,
    billing_provider_id         INTEGER         NOT NULL,
    claim_type                  VARCHAR(15),        -- Professional, Institutional, Dental, Pharmacy
    claim_status                VARCHAR(15),        -- Paid, Denied, Pending, Reversed
    is_preventative             BOOLEAN         NOT NULL DEFAULT FALSE,  -- true iff every line is a preventative procedure
    received_date               DATE            NOT NULL,
    service_date_start          DATE            NOT NULL,
    service_date_end            DATE            NOT NULL,
    paid_date                   DATE,
    total_billed_amount         NUMERIC(12,2)   DEFAULT 0,
    total_allowed_amount        NUMERIC(12,2)   DEFAULT 0,
    total_paid_amount           NUMERIC(12,2)   DEFAULT 0,
    total_member_resp_amount    NUMERIC(12,2)   DEFAULT 0
)
WITH (appendoptimized=true, orientation=column, compresstype=zstd, compresslevel=5)
DISTRIBUTED BY (claim_id);

-- ---------------------------------------------------------------------
-- claim_line: one row per service/procedure line on a claim
-- ---------------------------------------------------------------------
CREATE TABLE claims_demo.claim_line (
    claim_id                BIGINT          NOT NULL,
    line_number             SMALLINT        NOT NULL,
    procedure_id            INTEGER         NOT NULL,
    diagnosis_id            INTEGER         NOT NULL,
    rendering_provider_id   INTEGER         NOT NULL,
    service_date            DATE            NOT NULL,
    place_of_service        VARCHAR(5),
    modifier_code           VARCHAR(5),
    units                    SMALLINT        DEFAULT 1,
    billed_amount            NUMERIC(10,2)   DEFAULT 0,
    allowed_amount           NUMERIC(10,2)   DEFAULT 0,
    paid_amount              NUMERIC(10,2)   DEFAULT 0,
    copay_amount             NUMERIC(10,2)   DEFAULT 0,
    coinsurance_amount       NUMERIC(10,2)   DEFAULT 0,
    deductible_amount        NUMERIC(10,2)   DEFAULT 0,
    line_status               VARCHAR(15)        -- Paid, Denied
)
WITH (appendoptimized=true, orientation=column, compresstype=zstd, compresslevel=5)
DISTRIBUTED BY (claim_id);

Running query in 'postgresql://gpadmin@cdw:5432/dev'

++
||
++
++

## Generate History
Generate some data to seed the EDW with history.

In [4]:
%%sql
-- =====================================================================
-- Medical Claims Demo — Sample Data
-- =====================================================================

SET search_path TO claims_demo, public;
SELECT setseed(0.4218);

-- ---------------------------------------------------------------------
-- groups (10)
-- ---------------------------------------------------------------------
INSERT INTO groups (group_id, group_name, group_type, funding_type, industry_sic, state, effective_date, termination_date) VALUES
(1,  'Acme Manufacturing Inc.',        'Employer',     'Fully Insured', '3462', 'OH', '2022-01-01', NULL),
(2,  'Blue River Logistics',          'Employer',     'Self Funded',   '4213', 'TX', '2021-06-01', NULL),
(3,  'Cascade Health Systems',        'Employer',     'Self Funded',   '8062', 'WA', '2020-01-01', NULL),
(4,  'Delta Retail Group',            'Employer',     'Fully Insured', '5311', 'GA', '2023-03-01', NULL),
(5,  'Evergreen School District',     'Government',   'Fully Insured', '8211', 'OR', '2019-09-01', NULL),
(6,  'Falcon Financial Partners',     'Employer',     'Fully Insured', '6199', 'NY', '2022-01-01', NULL),
(7,  'Granite State University',      'Government',   'Self Funded',   '8221', 'NH', '2018-08-01', NULL),
(8,  'Harbor Point Insurance Assoc.', 'Association',  'Fully Insured', '6411', 'MA', '2021-01-01', NULL),
(9,  'Ironwood Construction Co.',     'Employer',     'Fully Insured', '1542', 'CO', '2023-01-01', NULL),
(10, 'Individual Marketplace Plans',  'Individual',   'Fully Insured', NULL,   'FL', '2020-01-01', NULL);

-- ---------------------------------------------------------------------
-- members (200000) — random but deterministic (setseed above)
-- ---------------------------------------------------------------------
INSERT INTO members (member_id, group_id, subscriber_id, first_name, last_name, date_of_birth, gender,
                      relationship_code, plan_type, address_line1, city, state, zip_code,
                      effective_date, termination_date)
SELECT
    s.member_id,
    (1 + floor(random()*10))::int AS group_id,
    'SUB' || lpad(((s.member_id - 1) / 3 + 1)::text, 6, '0') AS subscriber_id,
    (ARRAY['James','Mary','John','Patricia','Robert','Jennifer','Michael','Linda','William','Elizabeth',
           'David','Barbara','Richard','Susan','Joseph','Jessica','Carlos','Maria','Wei','Priya'])
        [(1 + floor(random()*20))::int] AS first_name,
    (ARRAY['Smith','Johnson','Williams','Brown','Jones','Garcia','Miller','Davis','Rodriguez','Martinez',
           'Wilson','Anderson','Taylor','Thomas','Moore','Jackson','Lee','Patel','Nguyen','Kim'])
        [(1 + floor(random()*20))::int] AS last_name,
    (date '1945-01-01' + (floor(random()*21900))::int) AS date_of_birth,
    (ARRAY['M','F'])[(1 + floor(random()*2))::int] AS gender,
    (ARRAY['Subscriber','Spouse','Dependent','Dependent'])[(1 + floor(random()*4))::int] AS relationship_code,
    (ARRAY['HMO','PPO','EPO','HDHP'])[(1 + floor(random()*4))::int] AS plan_type,
    (100 + floor(random()*9899))::text || ' Main St' AS address_line1,
    (ARRAY['Columbus','Austin','Seattle','Atlanta','Portland','Albany','Manchester','Boston','Denver','Miami'])
        [(1 + floor(random()*10))::int] AS city,
    (ARRAY['OH','TX','WA','GA','OR','NY','NH','MA','CO','FL'])[(1 + floor(random()*10))::int] AS state,
    lpad((10000 + floor(random()*89999))::text, 5, '0') AS zip_code,
    (date '2022-01-01' + (floor(random()*365))::int) AS effective_date,
    NULL AS termination_date
FROM generate_series(1, 200000) AS s(member_id);

-- ---------------------------------------------------------------------
-- providers (5000)
-- ---------------------------------------------------------------------
INSERT INTO providers (provider_id, npi, provider_name, provider_type, taxonomy_code, specialty_desc,
                        tax_id, address_line1, city, state, zip_code, network_status, effective_date)
SELECT
    s.provider_id,
    lpad((1000000000 + s.provider_id)::text, 10, '0') AS npi,
    spec.specialty_desc || ' Associates of ' ||
        (ARRAY['Columbus','Austin','Seattle','Atlanta','Portland','Albany','Manchester','Boston','Denver','Miami'])
            [(1 + floor(random()*10))::int] AS provider_name,
    (ARRAY['Individual','Individual','Facility','Group'])[(1 + floor(random()*4))::int] AS provider_type,
    spec.taxonomy_code,
    spec.specialty_desc,
    lpad((100000000 + s.provider_id)::text, 9, '0') AS tax_id,
    (200 + floor(random()*8899))::text || ' Medical Pkwy' AS address_line1,
    (ARRAY['Columbus','Austin','Seattle','Atlanta','Portland','Albany','Manchester','Boston','Denver','Miami'])
        [(1 + floor(random()*10))::int] AS city,
    (ARRAY['OH','TX','WA','GA','OR','NY','NH','MA','CO','FL'])[(1 + floor(random()*10))::int] AS state,
    lpad((10000 + floor(random()*89999))::text, 5, '0') AS zip_code,
    (CASE WHEN random() < 0.85 THEN 'In-Network' ELSE 'Out-of-Network' END) AS network_status,
    (date '2019-01-01' + (floor(random()*1800))::int) AS effective_date
FROM generate_series(1, 5000) AS s(provider_id)
CROSS JOIN LATERAL (
    SELECT * FROM (VALUES
        ('207Q00000X', 'Family Medicine'),
        ('207R00000X', 'Internal Medicine'),
        ('207RC0000X', 'Cardiovascular Disease'),
        ('208D00000X', 'General Practice'),
        ('2084P0800X', 'Psychiatry'),
        ('363L00000X', 'Nurse Practitioner'),
        ('207X00000X', 'Orthopaedic Surgery'),
        ('261QP2300X', 'Physical Therapy Clinic'),
        ('282N00000X', 'General Acute Care Hospital'),
        ('367500000X', 'Radiology')
    ) AS t(taxonomy_code, specialty_desc)
    ORDER BY random() LIMIT 1
) spec;

-- ---------------------------------------------------------------------
-- diagnosis (15 ICD-10-CM reference codes)
-- ---------------------------------------------------------------------
INSERT INTO diagnosis (diagnosis_id, diagnosis_code, diagnosis_desc, diagnosis_category, chronic_flag) VALUES
(1,  'E11.9',    'Type 2 diabetes mellitus without complications',                'Endocrine',              TRUE),
(2,  'I10',      'Essential (primary) hypertension',                             'Circulatory',            TRUE),
(3,  'J06.9',    'Acute upper respiratory infection, unspecified',               'Respiratory',            FALSE),
(4,  'M54.5',    'Low back pain',                                                'Musculoskeletal',        FALSE),
(5,  'K21.9',    'Gastro-esophageal reflux disease without esophagitis',         'Digestive',              TRUE),
(6,  'F41.9',    'Anxiety disorder, unspecified',                                'Mental Health',          TRUE),
(7,  'N39.0',    'Urinary tract infection, site not specified',                  'Genitourinary',          FALSE),
(8,  'J45.909',  'Unspecified asthma, uncomplicated',                            'Respiratory',            TRUE),
(9,  'E78.5',    'Hyperlipidemia, unspecified',                                  'Endocrine',              TRUE),
(10, 'M25.50',   'Pain in unspecified joint',                                    'Musculoskeletal',        FALSE),
(11, 'R51',      'Headache',                                                     'Nervous System',         FALSE),
(12, 'Z00.00',   'Encounter for general adult medical exam w/o abnormal findings','Factors Influencing Health', FALSE),
(13, 'O80',      'Encounter for full-term uncomplicated delivery',               'Pregnancy/Childbirth',   FALSE),
(14, 'S52.501A', 'Fracture of lower end of radius, unspecified, initial encounter','Injury',               FALSE),
(15, 'C50.911',  'Malignant neoplasm of unspecified site of right female breast','Neoplasms',              TRUE);

-- ---------------------------------------------------------------------
-- procedures (16 CPT/HCPCS reference codes)
-- ---------------------------------------------------------------------
INSERT INTO procedures (procedure_id, procedure_code, procedure_desc, procedure_category, code_type, is_preventative) VALUES
(1,  '99213', 'Office/outpatient visit, established patient, low complexity',   'Evaluation & Management', 'CPT',   FALSE),
(2,  '99214', 'Office/outpatient visit, established patient, moderate complexity','Evaluation & Management','CPT',  FALSE),
(3,  '99203', 'Office/outpatient visit, new patient, low complexity',           'Evaluation & Management', 'CPT',   FALSE),
(4,  '80053', 'Comprehensive metabolic panel',                                  'Laboratory',              'CPT',   FALSE),
(5,  '85025', 'Complete blood count (CBC) with differential',                   'Laboratory',              'CPT',   FALSE),
(6,  '93000', 'Electrocardiogram, routine ECG with interpretation',             'Diagnostic',              'CPT',   FALSE),
(7,  '71046', 'Chest X-ray, 2 views',                                           'Radiology',               'CPT',   FALSE),
(8,  '73721', 'MRI, lower extremity joint',                                     'Radiology',               'CPT',   FALSE),
(9,  '29881', 'Knee arthroscopy with meniscectomy',                             'Surgery',                 'CPT',   FALSE),
(10, '45378', 'Colonoscopy, diagnostic',                                        'Surgery',                 'CPT',   FALSE),
(11, '90834', 'Psychotherapy, 45 minutes',                                      'Behavioral Health',       'CPT',   FALSE),
(12, 'J1100', 'Injection, dexamethasone sodium phosphate, 1 mg',                'Drug/Injection',          'HCPCS', FALSE),
(13, 'G0439', 'Annual wellness visit, includes personalized prevention plan',   'Preventive',              'HCPCS', TRUE),
(14, '99284', 'Emergency department visit, moderate severity',                  'Evaluation & Management', 'CPT',   FALSE),
(15, '27447', 'Total knee arthroplasty',                                        'Surgery',                 'CPT',   FALSE),
(16, '90471', 'Immunization administration (one vaccine)',                      'Preventive',              'CPT',   TRUE);

-- ---------------------------------------------------------------------
-- claim_header (1,500,000) — totals populated after claim_line is loaded
-- ---------------------------------------------------------------------
INSERT INTO claim_header (claim_id, member_id, group_id, billing_provider_id, claim_type, claim_status,
                           received_date, service_date_start, service_date_end, paid_date,
                           total_billed_amount, total_allowed_amount, total_paid_amount, total_member_resp_amount)
SELECT
    c.claim_id,
    c.member_id,
    mem.group_id,
    c.billing_provider_id,
    c.claim_type,
    c.claim_status,
    c.service_date_start + (2 + floor(random()*5))::int  AS received_date,
    c.service_date_start,
    c.service_date_start + floor(random()*2)::int          AS service_date_end,
    CASE WHEN c.claim_status = 'Paid'
         THEN c.service_date_start + (10 + floor(random()*25))::int
         ELSE NULL END                                       AS paid_date,
    0, 0, 0, 0
FROM (
    SELECT
        s AS claim_id,
        (1 + floor(random()*200))::int AS member_id,
        (1 + floor(random()*50))::int  AS billing_provider_id,
        (ARRAY['Professional','Professional','Professional','Institutional','Dental'])
            [(1 + floor(random()*5))::int]     AS claim_type,
        (CASE WHEN random() < 0.80 THEN 'Paid'
              WHEN random() < 0.92 THEN 'Denied'
              ELSE 'Pending' END)        AS claim_status,
        (date '2024-01-01' + (floor(random()*620))::int) AS service_date_start
    FROM generate_series(1, 1500000) AS s
) c
JOIN members mem ON mem.member_id = c.member_id;

-- ---------------------------------------------------------------------
-- claim_line (~3,000, 1-3 lines per claim)
-- ---------------------------------------------------------------------
-- ~20% of claims are generated as pure preventative visits (checkup
-- and/or vaccine only: procedure 13 G0439 or 16 90471, diagnosis 12
-- Z00.00 wellness exam), lower cost, and fully covered ($0 member
-- responsibility when Paid) -- consistent with how preventative care is
-- typically billed. claim_header.is_preventative is derived afterward
-- from the actual lines generated (true iff every line's procedure is
-- flagged is_preventative), not from this branch directly, so it stays
-- correct even on the rare chance a "regular" claim's random draw lands
-- entirely on a preventative code too.
WITH claim_meta AS (
    -- Line count and visit type are picked here as ordinary projected
    -- columns (not as a generate_series() bound or inline in a CASE
    -- feeding one). A volatile expression that doesn't reference any
    -- column of the driving table can be evaluated once and reused for
    -- every row in a LATERAL FROM-item, which would silently give every
    -- claim the same number/type of lines.
    SELECT
        claim_id,
        billing_provider_id,
        service_date_start,
        claim_status,
        is_preventative_visit,
        CASE WHEN is_preventative_visit THEN preventative_line_count ELSE regular_line_count END AS line_count
    FROM (
        SELECT
            claim_id, billing_provider_id, service_date_start, claim_status,
            (random() < 0.20)               AS is_preventative_visit,
            (1 + floor(random()*2))::int    AS preventative_line_count,
            (1 + floor(random()*3))::int    AS regular_line_count
        FROM claim_header
    ) x
),
base AS (
    SELECT
        cm.claim_id,
        ln.line_number,
        cm.is_preventative_visit,
        CASE WHEN cm.is_preventative_visit
             THEN (ARRAY[13, 16])[(1 + floor(random()*2))::int]    -- G0439 wellness visit or 90471 immunization
             ELSE (1 + floor(random()*15))::int
        END                                 AS procedure_id,
        CASE WHEN cm.is_preventative_visit
             THEN 12                                                -- Z00.00 general wellness exam
             ELSE (1 + floor(random()*15))::int
        END                                 AS diagnosis_id,
        cm.billing_provider_id             AS rendering_provider_id,
        cm.service_date_start              AS service_date,
        (ARRAY['11','21','22','23','19'])[(1 + floor(random()*5))::int] AS place_of_service,
        CASE WHEN random() < 0.15
             THEN (ARRAY['25','59','LT','RT'])[(1 + floor(random()*4))::int]
             ELSE NULL END                  AS modifier_code,
        (1 + floor(random()*3))::int        AS units,
        cm.claim_status,
        CASE WHEN cm.is_preventative_visit
             THEN round((30 + random()*170)::numeric, 2)            -- wellness visit / vaccine: lower cost
             ELSE round((50 + random()*950)::numeric, 2)
        END                                 AS billed_amount
    FROM claim_meta cm
    CROSS JOIN LATERAL generate_series(1, cm.line_count) AS ln(line_number)
),
amounts AS (
    SELECT *,
           CASE WHEN is_preventative_visit THEN billed_amount        -- fully allowed, no network discount
                ELSE round((billed_amount * (0.55 + random()*0.35))::numeric, 2)
           END AS allowed_amount
    FROM base
),
paid AS (
    SELECT *,
           CASE WHEN claim_status IN ('Denied','Pending') THEN 0
                WHEN is_preventative_visit THEN allowed_amount        -- $0 member cost-share
                ELSE round((allowed_amount * (0.70 + random()*0.30))::numeric, 2)
           END AS paid_amount
    FROM amounts
),
resp AS (
    SELECT *,
           greatest(allowed_amount - paid_amount, 0) AS member_resp
    FROM paid
)
INSERT INTO claim_line (claim_id, line_number, procedure_id, diagnosis_id, rendering_provider_id,
                         service_date, place_of_service, modifier_code, units,
                         billed_amount, allowed_amount, paid_amount,
                         copay_amount, coinsurance_amount, deductible_amount, line_status)
SELECT
    claim_id, line_number, procedure_id, diagnosis_id, rendering_provider_id,
    service_date, place_of_service, modifier_code, units,
    billed_amount, allowed_amount, paid_amount,
    round((member_resp * 0.3)::numeric, 2) AS copay_amount,
    round((member_resp * 0.4)::numeric, 2) AS coinsurance_amount,
    round((member_resp * 0.3)::numeric, 2) AS deductible_amount,
    CASE WHEN claim_status = 'Denied' THEN 'Denied' ELSE 'Paid' END AS line_status
FROM resp;

-- ---------------------------------------------------------------------
-- Roll claim_line amounts (and the preventative flag) up into claim_header
-- ---------------------------------------------------------------------
UPDATE claim_header ch
SET is_preventative = agg.is_preventative
FROM (
    SELECT cl.claim_id, bool_and(p.is_preventative) AS is_preventative
    FROM claim_line cl
    JOIN procedures p ON p.procedure_id = cl.procedure_id
    GROUP BY cl.claim_id
) agg
WHERE ch.claim_id = agg.claim_id;

UPDATE claim_header ch
SET total_billed_amount      = agg.billed,
    total_allowed_amount     = agg.allowed,
    total_paid_amount        = agg.paid,
    total_member_resp_amount = agg.member_resp
FROM (
    SELECT claim_id,
           sum(billed_amount)                                    AS billed,
           sum(allowed_amount)                                   AS allowed,
           sum(paid_amount)                                      AS paid,
           sum(copay_amount + coinsurance_amount + deductible_amount) AS member_resp
    FROM claim_line
    GROUP BY claim_id
) agg
WHERE ch.claim_id = agg.claim_id;


Running query in 'postgresql://gpadmin@cdw:5432/dev'

1 rows affected.

10 rows affected.

200000 rows affected.

5000 rows affected.

...

1500000 rows affected.

2848775 rows affected.

1500000 rows affected.

1500000 rows affected.

++
||
++
++

## Stage schema
The stage schema will have a table that uses PGAA to query Iceberg. It will also have a small table that tracks the max event_id records that have been loaded to WarehousePG.

This table matches the table in PostgreSQL and also Iceberg.

In [5]:
%%sql
DROP SCHEMA IF EXISTS stage CASCADE;

CREATE SCHEMA stage;

CREATE TABLE stage.claim_events (
    -- ---- event metadata -------------------------------------------------
    event_id                            BIGINT,
    event_type                          VARCHAR(10),  -- INSERT, UPDATE, DELETE
    event_ts                            TIMESTAMPTZ,

    -- ---- claim_line grain identifiers ------------------------------------
    claim_id                            BIGINT,
    line_number                         SMALLINT,

    -- ---- claim_header ------------------------------------------------
    claim_type                          VARCHAR(15),        -- Professional, Institutional, Dental, Pharmacy
    claim_status                        VARCHAR(15),        -- Paid, Denied, Pending, Reversed
    claim_is_preventative               BOOLEAN,            -- true iff every line is a preventative procedure (checkup, vaccine)
    received_date                       DATE,
    service_date_start                  DATE,
    service_date_end                    DATE,
    paid_date                           DATE,
    claim_total_billed_amount           NUMERIC(12,2),
    claim_total_allowed_amount          NUMERIC(12,2),
    claim_total_paid_amount             NUMERIC(12,2),
    claim_total_member_resp_amount      NUMERIC(12,2),

    -- ---- claim_line --------------------------------------------------
    line_service_date                   DATE,
    place_of_service                    VARCHAR(5),
    modifier_code                       VARCHAR(5),
    units                                SMALLINT,
    line_billed_amount                  NUMERIC(10,2),
    line_allowed_amount                 NUMERIC(10,2),
    line_paid_amount                    NUMERIC(10,2),
    line_copay_amount                   NUMERIC(10,2),
    line_coinsurance_amount             NUMERIC(10,2),
    line_deductible_amount              NUMERIC(10,2),
    line_status                          VARCHAR(15),       -- Paid, Denied

    -- ---- members (the insured person) --------------------------------
    member_id                           INTEGER,
    member_subscriber_id                VARCHAR(20),
    member_first_name                   VARCHAR(50),
    member_last_name                    VARCHAR(50),
    member_date_of_birth                DATE,
    member_gender                       CHAR(1),
    member_relationship_code            VARCHAR(15),        -- Subscriber, Spouse, Dependent
    member_plan_type                    VARCHAR(10),        -- HMO, PPO, EPO, HDHP
    member_address_line1                VARCHAR(100),
    member_city                         VARCHAR(50),
    member_state                        CHAR(2),
    member_zip_code                     VARCHAR(10),
    member_effective_date               DATE,
    member_termination_date             DATE,

    -- ---- groups (insurance group / plan sponsor) ----------------------
    group_id                            INTEGER,
    group_name                          VARCHAR(100),
    group_type                          VARCHAR(30),        -- Employer, Individual, Government, Association
    group_funding_type                  VARCHAR(20),        -- Fully Insured, Self Funded
    group_industry_sic                  VARCHAR(10),
    group_state                         CHAR(2),
    group_effective_date                DATE,
    group_termination_date              DATE,

    -- ---- providers: billing provider on the claim ---------------------
    billing_provider_id                 INTEGER,
    billing_provider_npi                VARCHAR(10),
    billing_provider_name               VARCHAR(100),
    billing_provider_type               VARCHAR(20),        -- Individual, Facility, Group
    billing_provider_taxonomy_code      VARCHAR(10),
    billing_provider_specialty_desc     VARCHAR(60),
    billing_provider_state              CHAR(2),
    billing_provider_network_status     VARCHAR(15),        -- In-Network, Out-of-Network

    -- ---- providers: rendering provider on this line --------------------
    rendering_provider_id               INTEGER,
    rendering_provider_npi              VARCHAR(10),
    rendering_provider_name             VARCHAR(100),
    rendering_provider_specialty_desc   VARCHAR(60),
    rendering_provider_state            CHAR(2),
    rendering_provider_network_status   VARCHAR(15),

    -- ---- diagnosis on this line ----------------------------------------
    diagnosis_id                        INTEGER,
    diagnosis_code                      VARCHAR(10),        -- ICD-10-CM
    diagnosis_desc                      VARCHAR(255),
    diagnosis_category                  VARCHAR(60),
    diagnosis_chronic_flag              BOOLEAN,

    -- ---- procedure on this line -----------------------------------------
    procedure_id                        INTEGER,
    procedure_code                      VARCHAR(10),        -- CPT / HCPCS
    procedure_desc                      VARCHAR(255),
    procedure_category                  VARCHAR(60),
    procedure_code_type                 VARCHAR(10),        -- CPT, HCPCS
    procedure_is_preventative           BOOLEAN            -- true for wellness visits, immunizations, etc.

)
USING PGAA
WITH (
  pgaa.format = 'iceberg',
  pgaa.storage_location = 'pgaa-demo',
  pgaa.path = 'claims_source.claim_events'
)
DISTRIBUTED RANDOMLY;

CREATE TABLE stage.claims_event_log (event_id bigint, stage_timestamp timestamp default now()) DISTRIBUTED REPLICATED;
INSERT INTO stage.claims_event_log VALUES (-1);

Running query in 'postgresql://gpadmin@cdw:5432/dev'

1 rows affected.

++
||
++
++

## Incremental Load
- As data is inserted in PostgreSQL, PGAA replicates the data to Iceberg.
- In WarehousePG, we can load that data into WarehousePG tables by querying Iceberg directly.
- Using a function here to simplify executing this step in a repeated way.

In [6]:
%%sql

CREATE OR REPLACE FUNCTION stage.fn_load_incremental() RETURNS void AS
$$
BEGIN
    -- ---------------------------------------------------------------------
    -- groups
    -- ---------------------------------------------------------------------
    INSERT INTO claims_demo.groups
        (group_id, group_name, group_type, funding_type, industry_sic, state, effective_date, termination_date)
    SELECT DISTINCT ON (group_id)
        group_id,
        group_name,
        group_type,
        group_funding_type,
        group_industry_sic,
        group_state,
        COALESCE(group_effective_date, CURRENT_DATE),   -- stage doesn't always carry this; NOT NULL downstream
        group_termination_date
    FROM stage.claim_events
    WHERE event_id > (SELECT MAX(event_id) FROM stage.claims_event_log)
    AND group_id NOT IN (SELECT group_id FROM claims_demo.groups);

    -- ---------------------------------------------------------------------
    -- members
    -- ---------------------------------------------------------------------
    INSERT INTO claims_demo.members
        (member_id, group_id, subscriber_id, first_name, last_name, date_of_birth, gender,
         relationship_code, plan_type, address_line1, city, state, zip_code,
         effective_date, termination_date)
    SELECT DISTINCT ON (member_id)
        member_id,
        group_id,
        member_subscriber_id,
        member_first_name,
        member_last_name,
        member_date_of_birth,
        member_gender,
        member_relationship_code,
        member_plan_type,
        member_address_line1,
        member_city,
        member_state,
        member_zip_code,
        COALESCE(member_effective_date, CURRENT_DATE),  -- NOT NULL downstream
        member_termination_date
    FROM stage.claim_events
    WHERE event_id > (SELECT MAX(event_id) FROM stage.claims_event_log)
    AND member_id NOT IN (SELECT member_id FROM claims_demo.members);
    
    -- ---------------------------------------------------------------------
    -- providers
    -- ---------------------------------------------------------------------
    -- A provider can show up as billing_provider_* on some lines and
    -- rendering_provider_* on others (only billing carries provider_type /
    -- taxonomy_code in this feed). Union both roles, then prefer the
    -- billing-side attributes for a given provider_id when both exist.
    WITH provider_roles AS (
        SELECT
            billing_provider_id            AS provider_id,
            billing_provider_npi           AS npi,
            billing_provider_name          AS provider_name,
            billing_provider_type          AS provider_type,
            billing_provider_taxonomy_code AS taxonomy_code,
            billing_provider_specialty_desc AS specialty_desc,
            billing_provider_state         AS state,
            billing_provider_network_status AS network_status,
            1 AS role_priority                          -- billing wins ties
        FROM stage.claim_events
        WHERE event_id > (SELECT MAX(event_id) FROM stage.claims_event_log)
        UNION ALL
        SELECT
            rendering_provider_id,
            rendering_provider_npi,
            rendering_provider_name,
            NULL,                                        -- not carried for rendering-only rows
            NULL,
            rendering_provider_specialty_desc,
            rendering_provider_state,
            rendering_provider_network_status,
            2
        FROM stage.claim_events
        WHERE event_id > (SELECT MAX(event_id) FROM stage.claims_event_log)
    )
    INSERT INTO claims_demo.providers
        (provider_id, npi, provider_name, provider_type, taxonomy_code, specialty_desc,
         tax_id, address_line1, city, state, zip_code, network_status, effective_date)
    SELECT DISTINCT ON (provider_id)
        provider_id,
        npi,
        provider_name,
        provider_type,
        taxonomy_code,
        specialty_desc,
        NULL,           -- tax_id: not carried in stage
        NULL,           -- address_line1: not carried in stage
        NULL,           -- city: not carried in stage
        state,
        NULL,           -- zip_code: not carried in stage
        network_status,
        CURRENT_DATE    -- effective_date: not carried in stage; NOT NULL downstream
    FROM provider_roles
    WHERE provider_id NOT IN (select provider_id FROM provider_roles);
    
    -- ---------------------------------------------------------------------
    -- claim_header (one row per claim_id; header columns repeat identically
    -- across every line of a claim in stage, so DISTINCT ON collapses them)
    -- ---------------------------------------------------------------------
    INSERT INTO claims_demo.claim_header
        (claim_id, member_id, group_id, billing_provider_id, claim_type, claim_status, is_preventative,
         received_date, service_date_start, service_date_end, paid_date,
         total_billed_amount, total_allowed_amount, total_paid_amount, total_member_resp_amount)
    SELECT DISTINCT ON (claim_id)
        claim_id,
        member_id,
        group_id,
        billing_provider_id,
        claim_type,
        claim_status,
        claim_is_preventative,
        received_date,
        service_date_start,
        service_date_end,
        paid_date,
        claim_total_billed_amount,
        claim_total_allowed_amount,
        claim_total_paid_amount,
        claim_total_member_resp_amount
    FROM stage.claim_events
    WHERE event_id > (SELECT MAX(event_id) FROM stage.claims_event_log);
    
    -- ---------------------------------------------------------------------
    -- claim_line (already at claim_line grain -- one stage row per line)
    -- ---------------------------------------------------------------------
    INSERT INTO claims_demo.claim_line
        (claim_id, line_number, procedure_id, diagnosis_id, rendering_provider_id, service_date,
         place_of_service, modifier_code, units, billed_amount, allowed_amount, paid_amount,
         copay_amount, coinsurance_amount, deductible_amount, line_status)
    SELECT
        claim_id,
        line_number,
        procedure_id,
        diagnosis_id,
        rendering_provider_id,
        line_service_date,
        place_of_service,
        modifier_code,
        units,
        line_billed_amount,
        line_allowed_amount,
        line_paid_amount,
        line_copay_amount,
        line_coinsurance_amount,
        line_deductible_amount,
        line_status
    FROM stage.claim_events
    WHERE event_id > (SELECT MAX(event_id) FROM stage.claims_event_log);
    
    INSERT INTO stage.claims_event_log SELECT MAX(event_id) FROM stage.claim_events;
END;
$$
LANGUAGE plpgsql;

Running query in 'postgresql://gpadmin@cdw:5432/dev'

++
||
++
++

## Load Incremental

In [15]:
%%sql
SELECT stage.fn_load_incremental();
SELECT COUNT(*) FROM claims_demo.members;

Running query in 'postgresql://gpadmin@cdw:5432/dev'

1 rows affected.

1 rows affected.

count
200167


## Simple Analytics
Simple anlaytical query showing the number of claims for members.

In [16]:
%%sql
SELECT
    m.member_id,
    m.first_name,
    m.last_name,
    g.group_name,
    count(ch.claim_id)              AS claim_count,
    sum(ch.total_billed_amount)     AS total_billed,
    sum(ch.total_paid_amount)       AS total_paid
FROM claims_demo.members m
JOIN claims_demo.claim_header ch ON ch.member_id = m.member_id
JOIN claims_demo.groups g ON g.group_id = m.group_id
GROUP BY m.member_id, m.first_name, m.last_name, g.group_name
ORDER BY total_paid DESC
LIMIT 20;

Running query in 'postgresql://gpadmin@cdw:5432/dev'

20 rows affected.

member_id,first_name,last_name,group_name,claim_count,total_billed,total_paid
24,Elizabeth,Rodriguez,Acme Manufacturing Inc.,7561,6725846.80,3431238.39
87,Carlos,Jackson,Acme Manufacturing Inc.,7661,6751260.24,3420422.16
192,Jessica,Rodriguez,Cascade Health Systems,7563,6712789.05,3419735.09
50,Mary,Smith,Individual Marketplace Plans,7601,6725216.51,3414978.23
146,Elizabeth,Kim,Granite State University,7613,6785755.16,3413676.86
63,Richard,Moore,Ironwood Construction Co.,7598,6626420.39,3408476.85
26,Patricia,Jones,Harbor Point Insurance Assoc.,7705,6785834.12,3401081.87
110,Robert,Rodriguez,Evergreen School District,7600,6688522.91,3400559.89
170,Maria,Moore,Delta Retail Group,7525,6713095.10,3400227.17
4,William,Brown,Harbor Point Insurance Assoc.,7573,6650061.51,3397818.13


## Machine Learning
Create a ML model in-database using Apache MADlib.

In [9]:
%%sql

-- ---------------------------------------------------------------------
-- 1. Feature table: one row per member, built from their entire claim
--    history to date. Used for both training and scoring below.
-- ---------------------------------------------------------------------
-- NOTE on label design: "has the member EVER had a non-preventative claim"
-- (an existence check over unbounded history) mathematically saturates to
-- TRUE for almost everyone once a member has more than a few claims --
-- with preventative visits at ~20% of claims, the odds that ALL of a
-- member's claims happen to be preventative shrinks fast (roughly 0.2^N
-- for N claims), so a member with even 3-4 claims has under a 1% chance
-- of qualifying as "clean." That produces a training set that's nearly
-- all one class, and logistic regression trained on that will just
-- predict "true" confidently for everyone -- every score near 1,
-- regardless of features.
--
-- Fix: use a RELATIVE threshold instead of an absolute one. Flag members
-- whose rate of non-preventative claims is above the population median,
-- computed from this same data. That guarantees a genuine, close to
-- 50/50 split to train on -- "more of this member's care looks like real
-- treatment than is typical" -- instead of a near-constant label.
DROP TABLE IF EXISTS claims_demo.member_features;
CREATE TABLE claims_demo.member_features AS
WITH member_claim_stats AS (
    SELECT
        ch.member_id,
        count(*)                                       AS total_claims,
        count(*) FILTER (WHERE NOT ch.is_preventative) AS non_preventative_claims,
        sum(ch.total_billed_amount)                     AS total_billed,
        sum(ch.total_paid_amount)                       AS total_paid,
        avg(ch.total_billed_amount)                     AS avg_billed_per_claim,
        avg((ch.claim_status = 'Denied')::int)          AS denial_rate,
        max(ch.service_date_start)                      AS last_claim_date
    FROM claims_demo.claim_header ch
    GROUP BY ch.member_id
),
member_diag_stats AS (
    SELECT
        ch.member_id,
        count(DISTINCT cl.diagnosis_id)   AS distinct_diagnosis_count,
        avg(d.chronic_flag::int)          AS chronic_diagnosis_ratio
    FROM claims_demo.claim_header ch
    JOIN claims_demo.claim_line cl ON cl.claim_id = ch.claim_id
    JOIN claims_demo.diagnosis d   ON d.diagnosis_id = cl.diagnosis_id
    GROUP BY ch.member_id
),
member_ratios AS (
    SELECT
        m.member_id,
        CASE WHEN COALESCE(mcs.total_claims, 0) > 0
             THEN mcs.non_preventative_claims::numeric / mcs.total_claims
             ELSE NULL
        END AS non_preventative_ratio
    FROM claims_demo.members m
    LEFT JOIN member_claim_stats mcs ON mcs.member_id = m.member_id
),
median_ratio AS (
    SELECT percentile_cont(0.5) WITHIN GROUP (ORDER BY non_preventative_ratio) AS med
    FROM member_ratios
    WHERE non_preventative_ratio IS NOT NULL
)
SELECT
    m.member_id,
    COALESCE(mcs.total_claims, 0)                                          AS total_claims,
    COALESCE(mcs.total_billed, 0)                                          AS total_billed,
    COALESCE(mcs.total_paid, 0)                                            AS total_paid,
    COALESCE(mcs.avg_billed_per_claim, 0)                                  AS avg_billed_per_claim,
    COALESCE(mcs.denial_rate, 0)                                           AS denial_rate,
    COALESCE(mds.distinct_diagnosis_count, 0)                              AS distinct_diagnosis_count,
    COALESCE(mds.chronic_diagnosis_ratio, 0)                               AS chronic_diagnosis_ratio,
    (CURRENT_DATE - COALESCE(mcs.last_claim_date, m.effective_date))::int  AS days_since_last_claim,
    (CURRENT_DATE - m.effective_date)::int                                 AS member_tenure_days,
    COALESCE(mr.non_preventative_ratio, 0) > median_ratio.med              AS had_non_preventative_claim
FROM claims_demo.members m
LEFT JOIN member_claim_stats mcs ON mcs.member_id = m.member_id
LEFT JOIN member_diag_stats  mds ON mds.member_id = m.member_id
LEFT JOIN member_ratios      mr  ON mr.member_id = m.member_id
CROSS JOIN median_ratio;

Running query in 'postgresql://gpadmin@cdw:5432/dev'

200107 rows affected.

++
||
++
++

In [10]:
%%sql
-- ---------------------------------------------------------------------
-- 2. Train the model
-- ---------------------------------------------------------------------
-- Monetary features are log-transformed (ln(x + 1)) so their scale
-- doesn't swamp the 0-1 / small-integer features during IRLS fitting.
-- The leading 1 is the intercept term.
DROP TABLE IF EXISTS claims_demo.member_claim_model;
DROP TABLE IF EXISTS claims_demo.member_claim_model_summary;

SELECT madlib.logregr_train(
    'claims_demo.member_features',            -- source_table
    'claims_demo.member_claim_model',         -- out_table
    'had_non_preventative_claim',             -- dependent_varname (must be boolean)
    'ARRAY[1, total_claims, ln(total_billed + 1), ln(total_paid + 1), ln(avg_billed_per_claim + 1),
           denial_rate, distinct_diagnosis_count, chronic_diagnosis_ratio,
           days_since_last_claim, member_tenure_days]',   -- independent_varname
    NULL,                                     -- grouping_cols
    20,                                        -- max_iter
    'irls'                                     -- optimizer
);

-- Inspect the fitted coefficients / significance before trusting the
-- scores below -- large std_err or p_values near 1 mean that feature
-- isn't pulling its weight.
SELECT
    unnest(ARRAY['intercept', 'total_claims', 'ln_total_billed', 'ln_total_paid', 'ln_avg_billed_per_claim',
                 'denial_rate', 'distinct_diagnosis_count', 'chronic_diagnosis_ratio',
                 'days_since_last_claim', 'member_tenure_days'])   AS feature,
    unnest(coef)                                                    AS coefficient,
    unnest(std_err)                                                 AS std_err,
    unnest(p_values)                                                AS p_value,
    unnest(odds_ratios)                                             AS odds_ratio
FROM claims_demo.member_claim_model;



Running query in 'postgresql://gpadmin@cdw:5432/dev'

1 rows affected.

10 rows affected.

feature,coefficient,std_err,p_value,odds_ratio
intercept,-12.064079519155348,6.448398895552545,0.061363887375713314,5.762843630921832e-06
total_claims,0.0012806787190775587,0.0008741318896534113,0.1428973542454413,1.0012814991382621
ln_total_billed,-0.21771236867302624,1.4742013501880102,0.8825940777960832,0.8043567666009537
ln_total_paid,-1.735328326633418,1.1893912614662971,0.14456360906329146,0.176342292909233
ln_avg_billed_per_claim,3.6375851387130607,1.2257741764181644,0.0030015240260305247,37.99996120151345
denial_rate,-4.472435226762311,2.8814450702507552,0.12062631492676171,0.011419472956230362
distinct_diagnosis_count,0.353323277348213,0.20255885926064787,0.08110691285667462,1.4237913485342144
chronic_diagnosis_ratio,1.4262029905129607,1.8222377729893584,0.43382345304501607,4.162862716099076
days_since_last_claim,-0.007298582363345404,0.012490450580596755,0.5589962993128951,0.9927279876085725
member_tenure_days,0.003324681126202307,0.0014538585715432862,0.022207606148954836,1.0033302140084883


In [11]:
%%sql
-- Overall fit diagnostics
SELECT * FROM claims_demo.member_claim_model_summary;

Running query in 'postgresql://gpadmin@cdw:5432/dev'

1 rows affected.

method,source_table,out_table,dependent_varname,independent_varname,optimizer_params,num_all_groups,num_failed_groups,num_rows_processed,num_missing_rows_skipped,grouping_col
logregr,claims_demo.member_features,claims_demo.member_claim_model,had_non_preventative_claim,"ARRAY[1, total_claims, ln(total_billed + 1), ln(total_paid + 1), ln(avg_billed_per_claim + 1), denial_rate, distinct_diagnosis_count, chronic_diagnosis_ratio, days_since_last_claim, member_tenure_days]","optimizer=irls, max_iter=20, tolerance=0.0001",1,0,200107,0,None


In [12]:
%%sql
-- ---------------------------------------------------------------------
-- 3. Score every member: probability of a non-preventative claim, plus a
--    flag at a 0.5 threshold (tune this based on how many alerts is
--    workable).
-- ---------------------------------------------------------------------
DROP TABLE IF EXISTS claims_demo.member_risk_scores;
CREATE TABLE claims_demo.member_risk_scores AS
SELECT
    f.member_id,
    m.first_name,
    m.last_name,
    f.total_claims,
    f.total_billed,
    f.denial_rate,
    f.chronic_diagnosis_ratio,
    f.days_since_last_claim,
    madlib.logregr_predict_prob(
        model.coef,
        ARRAY[1, f.total_claims, ln(f.total_billed + 1), ln(f.total_paid + 1), ln(f.avg_billed_per_claim + 1),
              f.denial_rate, f.distinct_diagnosis_count, f.chronic_diagnosis_ratio,
              f.days_since_last_claim, f.member_tenure_days]
    )                                                          AS non_preventative_score,
    madlib.logregr_predict_prob(
        model.coef,
        ARRAY[1, f.total_claims, ln(f.total_billed + 1), ln(f.total_paid + 1), ln(f.avg_billed_per_claim + 1),
              f.denial_rate, f.distinct_diagnosis_count, f.chronic_diagnosis_ratio,
              f.days_since_last_claim, f.member_tenure_days]
    ) >= 0.5                                                    AS needs_scrutiny
FROM claims_demo.member_features f
JOIN claims_demo.members m ON m.member_id = f.member_id
CROSS JOIN claims_demo.member_claim_model model;

Running query in 'postgresql://gpadmin@cdw:5432/dev'

200107 rows affected.

++
||
++
++

In [13]:
%%sql
-- ---------------------------------------------------------------------
-- 4. Provider alert list: highest-risk members, with the provider(s)
--    who most recently treated them -- who's the one to actually alert.
-- ---------------------------------------------------------------------
-- Avoid a LATERAL-per-row nested loop (forces the planner to probe
-- claim_header/providers once per member -- fine on a single-node OLTP
-- box, bad on an MPP cluster). Instead, compute each member's most
-- recent claim with a window function in one pass over claim_header,
-- then join that back to member_risk_scores and providers as ordinary
-- set-based (redistribute/hash) joins.
WITH ranked_claims AS (
    SELECT
        ch.member_id,
        ch.billing_provider_id,
        ch.service_date_start,
        row_number() OVER (
            PARTITION BY ch.member_id
            ORDER BY ch.service_date_start DESC
        ) AS rn
    FROM claims_demo.claim_header ch
),
latest AS (
    SELECT member_id, billing_provider_id, service_date_start AS last_service_date
    FROM ranked_claims
    WHERE rn = 1
)
SELECT
    mrs.member_id,
    mrs.first_name,
    mrs.last_name,
    round(mrs.non_preventative_score::numeric, 3)  AS non_preventative_score,
    mrs.total_claims,
    round(mrs.denial_rate::numeric, 3) as denial_rate,
    p.provider_id,
    p.provider_name,
    p.specialty_desc,
    latest.last_service_date
FROM claims_demo.member_risk_scores mrs
JOIN latest                    ON latest.member_id = mrs.member_id
JOIN claims_demo.providers p   ON p.provider_id = latest.billing_provider_id
WHERE mrs.needs_scrutiny
ORDER BY mrs.non_preventative_score DESC
LIMIT 10;

Running query in 'postgresql://gpadmin@cdw:5432/dev'

10 rows affected.

member_id,first_name,last_name,non_preventative_score,total_claims,denial_rate,provider_id,provider_name,specialty_desc,last_service_date
26,Patricia,Jones,0.721,7705,0.186,47,Internal Medicine Associates of Miami,Internal Medicine,2025-09-11
168,Barbara,Miller,0.715,7663,0.188,9,Internal Medicine Associates of Manchester,Internal Medicine,2025-09-11
151,James,Johnson,0.706,7685,0.187,18,Internal Medicine Associates of Seattle,Internal Medicine,2025-09-11
142,Michael,Lee,0.698,7599,0.192,14,Internal Medicine Associates of Portland,Internal Medicine,2025-09-11
71,Richard,Williams,0.691,7627,0.170,13,Internal Medicine Associates of Manchester,Internal Medicine,2025-09-11
39,Joseph,Jackson,0.690,7541,0.187,21,Internal Medicine Associates of Miami,Internal Medicine,2025-09-11
66,Joseph,Wilson,0.688,7554,0.186,35,Internal Medicine Associates of Columbus,Internal Medicine,2025-09-11
11,Maria,Jackson,0.685,7590,0.186,21,Internal Medicine Associates of Miami,Internal Medicine,2025-09-11
90,Jessica,Taylor,0.685,7642,0.183,26,Internal Medicine Associates of Columbus,Internal Medicine,2025-09-11
81,David,Rodriguez,0.677,7584,0.183,37,Internal Medicine Associates of Columbus,Internal Medicine,2025-09-11


## Low score members

In [14]:
%%sql

SELECT member_id, first_name, last_name, round(non_preventative_score::numeric, 2) as score, total_claims
FROM claims_demo.member_risk_scores
WHERE total_claims > 0
ORDER BY score ASC
LIMIT 10;


Running query in 'postgresql://gpadmin@cdw:5432/dev'

10 rows affected.

member_id,first_name,last_name,score,total_claims
5000086,Emma,Martinez,0.03,7
5000104,Liam,Jones,0.04,2
5000058,Elizabeth,Garcia,0.06,15
5000075,William,Johnson,0.09,8
5000054,Joseph,Thomas,0.11,12
5000110,Noah,Kim,0.11,1
5000021,David,Martinez,0.12,32
5000082,Jennifer,Lee,0.13,7
5000067,Richard,Smith,0.13,9
5000007,Sofia,Okafor,0.14,30
